# 03: Campaign Impact Analysis
This notebook evaluates how Wikimedia campaigns (Fundraising, Wiki Loves Monuments, etc.) influence Wikipedia usage behavior using an event-study framework.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import json
from datetime import timedelta

# Add src to path
sys.path.append(os.path.abspath('../../'))
from src.data_prep import clean_pageview_data, add_time_features

# Configuration
DATA_PATH = '../../data/raw/en_wiki_pageviews_daily.csv'
CONFIG_PATH = '../../config/campaign_dates.json'
REPORT_DIR = '../../reports/'
os.makedirs(REPORT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")

## 1. Load Data and Configuration

In [ ]:
# Load pageviews
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = clean_pageview_data(df)
df = add_time_features(df)

# Load campaign dates
with open(CONFIG_PATH, 'r') as f:
    campaign_config = json.load(f)

print(f"Data loaded from {DATA_PATH}")
print(f"Campaign types found: {list(campaign_config.keys())}")

## 2. Event Study Function
We will define a function to analyze a single campaign instance using an estimation window and an event window.

In [ ]:
def analyze_campaign(df, start_date, campaign_name, year):
    start_date = pd.to_datetime(start_date)
    
    # Windows
    est_start = start_date - timedelta(days=60)
    est_end = start_date - timedelta(days=31)
    event_start = start_date - timedelta(days=30)
    event_end = start_date + timedelta(days=30)
    
    # Data for windows
    est_data = df[(df['timestamp'] >= est_start) & (df['timestamp'] <= est_end)]
    event_data = df[(df['timestamp'] >= event_start) & (df['timestamp'] <= event_end)].copy()
    
    if est_data.empty or event_data.empty:
        return None
        
    # Baseline: Average per weekday in estimation window
    baseline = est_data.groupby('day_of_week')['views'].mean().to_dict()
    
    # Apply baseline
    event_data['expected_views'] = event_data['day_of_week'].map(baseline)
    event_data['excess_views'] = event_data['views'] - event_data['expected_views']
    event_data['percent_lift'] = (event_data['excess_views'] / event_data['expected_views']) * 100
    
    # Relative days
    event_data['relative_day'] = (event_data['timestamp'] - start_date).dt.days
    
    return event_data

# Example analysis (Fundraising 2023)
example_event = analyze_campaign(df, campaign_config['fundraising']['2023'], 'Fundraising', 2023)
example_event.head()

## 3. Visualization: Event Study Plot
Visualizing the traffic response around the 2023 fundraising campaign.

In [ ]:
def plot_event(event_data, title):
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=event_data, x='relative_day', y='percent_lift', marker='o', color='purple')
    
    # Reference lines
    plt.axvline(x=0, color='red', linestyle='--', label='Campaign Start')
    plt.axvspan(0, 14, color='orange', alpha=0.2, label='Active Campaign (14d)')
    plt.axhline(y=0, color='gray', linestyle='-')
    
    plt.title(title, fontsize=15)
    plt.xlabel('Days Relative to Start', fontsize=12)
    plt.ylabel('Pageview Lift (%)', fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.show()

if example_event is not None:
    plot_event(example_event, 'Wikipedia Fundraising 2023 Impact Analysis')
else:
    print("Not enough data for the example window.")

## 4. Cross-Campaign Comparison
Now we across multiple years and campaign types.

In [ ]:
all_results = []

for campaign_type, dates in campaign_config.items():
    for year, date_str in dates.items():
        event_df = analyze_campaign(df, date_str, campaign_type, year)
        if event_df is not None:
            # Compute metrics
            active_mask = (event_df['relative_day'] >= 0) & (event_df['relative_day'] <= 14)
            post_mask = (event_df['relative_day'] > 14)
            
            all_results.append({
                'Campaign': campaign_type,
                'Year': year,
                'Avg Lift (%)': event_df.loc[active_mask, 'percent_lift'].mean(),
                'Peak Lift (%)': event_df.loc[active_mask, 'percent_lift'].max(),
                'Post Residual (%)': event_df.loc[post_mask, 'percent_lift'].mean()
            })

comparison_df = pd.DataFrame(all_results)
comparison_df.sort_values(by='Avg Lift (%)', ascending=False).head(10)

## 5. Summary Visualization: Campaign Type Comparison

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=comparison_df, x='Campaign', y='Avg Lift (%)', palette='Set2')
plt.title('Average Traffic Lift by Campaign Type (2015-2025)', fontsize=15)
plt.ylabel('Average Campaign Lift (%)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'section_3_campaign_comparison.png'))
plt.show()